# The Gauntlet: Unsloth LLM Training & Evaluation

This notebook provides the complete setup to run `unsloth/Qwen2.5-1.5B-Instruct` against The Gauntlet environment. It handles loading the fast Unsloth model, parsing observations into prompts, extracting JSON actions, running the environment loop, and generating the reward curve plots for comparison against the baseline.

In [ ]:
# 1. Install required dependencies for Unsloth and environment
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install matplotlib aiosqlite

In [ ]:
# 2. Load the Unsloth Model
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model) # Enable native 2x faster inference

In [ ]:
# 3. Define the Agent Logic
import json
import re

SYSTEM_PROMPT = """You are an expert customer support triage agent in The Gauntlet environment.
You must classify tickets and respond to customers. Return ONLY a valid JSON object:
{
  "assign_priority": "Low" | "Medium" | "High" | "Critical",
  "assign_category": "Billing" | "Technical" | "Shipping" | "Security" | "Fraud" | "Compliance",
  "draft_response": "<professional reply to customer>",
  "escalate": true | false,
  "approve_refund": true | false | null
}

Read the system notice and ticket carefully and follow all active policy rules."""

def extract_json(text):
    """Attempt to extract JSON from the LLM output."""
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except:
            pass
    return None

def unsloth_agent(observation):
    """Generates an action using the Unsloth Qwen model."""
    obs_str = json.dumps(observation, indent=2)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Observation:\n{obs_str}\n\nAction JSON:"}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True)
    response_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    action = extract_json(response_text)
    if action is None:
        # Fallback if generation fails to produce valid JSON
        action = {
            "assign_priority": "Medium",
            "assign_category": "Technical",
            "draft_response": "Thank you for your message. We are looking into this.",
            "escalate": False,
            "approve_refund": None
        }
    return action

In [ ]:
# 4. Training / Evaluation Loop
import sys
import os
# Ensure the Gauntlet environment files are in the Colab working directory
if not os.path.exists('environment.py'):
    print("Please upload the environment files (environment.py, policy.py, rewards.py, etc.) to the Colab workspace.")

try:
    from environment import CustomerSupportEnv
except ImportError:
    print("Waiting for environment files...")

def run_llm_training_loop(num_episodes=20, task_id=2, drift_enabled=True, attacker_enabled=False):
    env = CustomerSupportEnv()
    episode_rewards = []
    episode_step_rewards = []
    difficulty_trace = []
    
    print(f"Starting training loop for {num_episodes} episodes...")
    for ep in range(num_episodes):
        obs = env.reset(task_id=task_id, drift_enabled=drift_enabled, attacker_enabled=attacker_enabled)
        done = False
        step_rewards = []
        
        while not done:
            # Note: For RL training (PPO/GRPO), you would collect logprobs and compute 
            # the policy gradient update here using TRL. For baseline evaluation, 
            # we just collect the rewards.
            action = unsloth_agent(obs)
            result = env.step(action)
            
            step_rewards.append(result["reward"])
            done = result["done"]
            if not done:
                obs = result["observation"]
                
        mean_r = sum(step_rewards) / max(len(step_rewards), 1)
        episode_rewards.append(mean_r)
        episode_step_rewards.extend(step_rewards)
        difficulty_trace.append(env.world_state.difficulty_level)
        
        print(f"Episode {ep+1}/{num_episodes} | Mean Reward: {mean_r:+.3f} | Difficulty: {env.world_state.difficulty_level:.3f}")
        
    return episode_rewards, episode_step_rewards, difficulty_trace

# Run the loop (start with a small number to test)
if os.path.exists('environment.py'):
    ep_r, step_r, diff_t = run_llm_training_loop(num_episodes=20)


In [ ]:
# 5. Plot the Results
import matplotlib.pyplot as plt

def plot_results(episode_rewards, step_rewards, difficulty_trace, title_suffix=""):
    fig, axes = plt.subplots(3, 1, figsize=(12, 14), facecolor="#1a1a2e")
    fig.suptitle(f"Qwen2.5-1.5B-Instruct Agent Performance{title_suffix}", fontsize=16, color="white", fontweight="bold")

    for ax in axes:
        ax.set_facecolor("#16213e")
        ax.tick_params(colors="white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.title.set_color("white")
        for spine in ax.spines.values():
            spine.set_color("#444")

    # 1. Episode mean reward
    ax1 = axes[0]
    ax1.plot(range(1, len(episode_rewards)+1), episode_rewards, color="#00d2ff", linewidth=1.5, alpha=0.7, label="Episode Reward")
    window = min(5, len(episode_rewards))
    if window > 1:
        rolling = [sum(episode_rewards[max(0,i-window):i+1]) / len(episode_rewards[max(0,i-window):i+1]) for i in range(len(episode_rewards))]
        ax1.plot(range(1, len(rolling)+1), rolling, color="#3a7bd5", linewidth=2.5, label=f"{window}-Episode Rolling Avg")
    ax1.axhline(y=0, color="#444", linestyle="--", linewidth=0.8)
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Mean Reward")
    ax1.set_title("Per-Episode Mean Reward")
    ax1.legend(facecolor="#16213e", edgecolor="#444", labelcolor="white")

    # 2. Step-level rewards
    ax2 = axes[1]
    ax2.scatter(range(len(step_rewards)), step_rewards, s=2, alpha=0.5, color="#00d2ff")
    step_window = min(20, len(step_rewards))
    if step_window > 1:
        step_rolling = [sum(step_rewards[max(0,i-step_window):i+1]) / len(step_rewards[max(0,i-step_window):i+1]) for i in range(len(step_rewards))]
        ax2.plot(range(len(step_rolling)), step_rolling, color="#3a7bd5", linewidth=1.5, label=f"{step_window}-Step Rolling Avg")
    ax2.axhline(y=0, color="#444", linestyle="--", linewidth=0.8)
    ax2.set_xlabel("Step (global)")
    ax2.set_ylabel("Reward")
    ax2.set_title("Per-Step Reward Distribution")
    ax2.legend(facecolor="#16213e", edgecolor="#444", labelcolor="white")

    # 3. Difficulty trace
    ax3 = axes[2]
    ax3.plot(range(1, len(difficulty_trace)+1), difficulty_trace, color="#b620e0", linewidth=2)
    ax3.set_xlabel("Episode")
    ax3.set_ylabel("Difficulty Level")
    ax3.set_title("Curriculum Difficulty Over Time")
    ax3.set_ylim(-0.05, 1.05)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

if os.path.exists('environment.py'):
    plot_results(ep_r, step_r, diff_t)
